# 데이터베이스 적재 준비
- 각 테이블에 code 컬럼 추가
- rel 테이블 데이터 준비

In [2]:
import pandas as pd

In [3]:
restaurant_df = pd.read_csv("db_csv_cat_tag/restaurant_df.csv")
review_df = pd.read_csv("db_csv_cat_tag/review_df.csv")
menu_df = pd.read_csv("db_csv_cat_tag/menu_df.csv")

category_df = pd.read_csv("db_csv_cat_tag/category_df.csv")
tag_df = pd.read_csv("db_csv_cat_tag/tag_df.csv")

## user 테이블 생성

In [4]:
user_df = pd.DataFrame(review_df["reviewer_name"].unique(), columns=["name"])
print(len(user_df))
user_df.head()

171


,name
0,냐밍
1,발통이
2,우리의나날
3,냠냠냠히
4,밍민


## _code 컬럼의 추가

In [5]:
# _code를 추가하고 맨 앞으로 빼줌
def add_code(df_name:str, prefix:str, df:pd.DataFrame) -> pd.DataFrame:
    columns = list(df.columns)

    codes = []
    for i in range(len(df)):
        l = len(str(i))
        buff = prefix + "0" * (4-l) + str(i)
        codes.append(buff)

    column_name = df_name + "_code"    
    df[column_name] = codes

    if column_name not in columns:
        columns = [column_name] + columns
        
    return df[columns]


In [6]:
restaurant_df = add_code("restaurant", "RES", restaurant_df)
restaurant_df.head()

,restaurant_code,restaurant_name,img_link,region,category,address,open_time,close_time,tel_no,tags,character,latitude,longitude
0,RES0000,유태우스시,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,"스시,회전초밥",서울특별시 동작구 보라매로 113 1층 유태우스시,11:00,22:00,0507-1358-1246,"데이트,혼밥","가성비좋은,서민적인,점심식사,저녁식사,바테이블,다찌석",37.499463,126.927897
1,RES0001,듀윗,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,"디저트,케이크",서울특별시 동작구 대방동1길 10 1층,08:00,21:00,0507-2093-6672,"데이트,기념일","예쁜,깔끔한,점심식사,테이크아웃,예약가능",37.500364,126.926712
2,RES0002,한제소곱창전골 신대방삼거리역점,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,"곱창전골,소곱창전골","서울특별시 동작구 여의대방로22길 121 상가동 1층 1444호, 1445호",16:00,00:00,0507-1359-9161,"술모임,해장용","지역주민이찾는,매콤한,저녁식사,예약가능,배달",37.499106,126.926924
3,RES0003,프로스퍼,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,"카페,커피",서울특별시 동작구 여의대방로24길 101 1층,10:00,00:00,0507-1361-3276,"차모임,혼자카페","캐주얼한,카공,점심식사,테이크아웃,와이파이",37.499088,126.925611
4,RES0004,일진아구찜,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,아구찜,서울특별시 동작구 여의대방로24길 93 1층,11:00,22:00,02-817-5554,가족외식,"서민적인,매콤한,점심식사,저녁식사,유아의자",37.498954,126.925189


In [7]:
user_df = add_code("user", "USR", user_df)

review_unique = review_df.drop_duplicates(subset="reviewer_name", keep="first")

result_df = user_df.merge(
    review_unique[
        ["reviewer_name", "reviewer_avg_score", "reviewer_rv_cnt", "reviewer_follower_cnt"]
    ],
    left_on="name",
    right_on="reviewer_name",
    how="left"
).drop(columns="reviewer_name")

user_df = result_df
user_df.head()

,user_code,name,reviewer_avg_score,reviewer_rv_cnt,reviewer_follower_cnt
0,USR0000,냐밍,4.3,355,69
1,USR0001,발통이,4.5,113,1
2,USR0002,우리의나날,4.5,420,2
3,USR0003,냠냠냠히,4.8,71,0
4,USR0004,밍민,4.8,232,2


In [8]:
review_df = add_code("review", "REV", review_df)

name_to_code = user_df.set_index("name")["user_code"]
review_df["reviewer_code"] = review_df["reviewer_name"].map(name_to_code)
review_df["restaurant_code"] = review_df["restaurant_code"].map(restaurant_df["restaurant_code"])

review_df = review_df.drop(columns=["reviewer_name", "reviewer_avg_score", "reviewer_rv_cnt", "reviewer_follower_cnt"])

header = ["review_code", "restaurant_code", "reviewer_code"]
cols = header + [c for c in list(review_df.columns) if c not in header]

review_df = review_df[cols]

review_df.head()

,review_code,restaurant_code,reviewer_code,reviewer_score,taste_level,price_level,service_level,review_content,review_menu,keywords
0,REV0000,RES0000,USR0000,4점,2.0,2.0,1.0,요즘 회전초밥집 정말 비싼데 여긴 가성비 갑이었어요! 위치는 조금 복잡한 곳에 있긴...,회전초밥,"저녁식사,식사모임,캐주얼한,이국적/이색적,가성비좋은"
1,REV0001,RES0000,USR0001,4점,1.0,2.0,2.0,한접시에 1500원이면 저렴해요 근데 맛도 생각보다 괜찮았어요! 엄청난 퀄리티는 아...,스시,"점심식사,지역주민이찾는"
2,REV0002,RES0000,USR0002,5점,2.0,2.0,2.0,생일날 갔어요. 저녁시간 웨이팅있었지만 금방 자리에 앉았어요. 회전초밥 모든접시가 ...,회전초밥 1500원,"저녁식사,데이트,서민적인,푸짐한,지역주민이찾는"
3,REV0003,RES0000,USR0003,5점,2.0,2.0,2.0,"모든접시 1,700원 이라는 가성비에 맛도 좋고 이런 굿굿!!","연어,장어","저녁식사,가성비좋은"
4,REV0004,RES0000,USR0004,5점,2.0,2.0,2.0,1500원이라가성비좋아요 다양하게 먹을 수 있고 여러가지종류있어서 좋아요,회전초밥,"점심식사,캐주얼한"


In [9]:
menu_df = add_code("menu", "MEN", menu_df)

menu_df["restaurant_code"] = menu_df["restaurant_code"].map(restaurant_df["restaurant_code"])

menu_df.head()

,menu_code,restaurant_code,menu_name,price,description
0,MEN0000,RES0000,회전초밥,NaN,NaN
1,MEN0001,RES0000,장어초밥,NaN,NaN
2,MEN0002,RES0001,프레지에 1호 홀케이크,50000.0,NaN
3,MEN0003,RES0001,프레지에,10500.0,NaN
4,MEN0004,RES0001,페르디셔 압펠,6000.0,NaN


In [10]:
category_df = add_code("category", "CAT", category_df)
category_df.head()

,category_code,category_name
0,CAT0000,고깃집
1,CAT0001,짜장면
2,CAT0002,일식당
3,CAT0003,숙성회
4,CAT0004,돼지갈비


In [11]:
tag_df = add_code("tag", "TAG", tag_df)
tag_df.head()

,tag_code,tag_name
0,TAG0000,24시간영업
1,TAG0001,회식
2,TAG0002,포장전문
3,TAG0003,이국적/이색적
4,TAG0004,지역주민이 찾는


## rel table 작성
- rel_restaurant_category
- rel_restaurant_tag
- rel_review_tag

In [12]:
def pipe(df:pd.DataFrame, col1, col2, data):
    return df.loc[df[col1] == data, col2].iloc[0]

In [13]:
# rel_restaurant_category
restaurants = []
categories = []

for res, cat in zip(restaurant_df["restaurant_code"], restaurant_df["category"]):
    cats = cat.split(",")
    for c in cats:
        restaurants.append(res)
        categories.append(pipe(category_df, "category_name", "category_code", c))

df_dict = {
    "restaurant_code": restaurants,
    "category_code": categories
}
rel_restaurant_category = pd.DataFrame(df_dict)

rel_restaurant_category.head()

,restaurant_code,category_code
0,RES0000,CAT0052
1,RES0000,CAT0095
2,RES0001,CAT0015
3,RES0001,CAT0007
4,RES0002,CAT0054


In [14]:
# rel_restaurant_tag
restaurants = []
tags = []

for res, cdata, tdata in zip(restaurant_df["restaurant_code"], restaurant_df["character"], restaurant_df["tags"]):
    tlist = ([] if pd.isna(cdata) else cdata.split(",")) + ([] if pd.isna(tdata) else tdata.split(","))
    for t in tlist:
        restaurants.append(res)
        tags.append(pipe(tag_df, "tag_name", "tag_code", t))

df_dict = {
    "restaurant_code": restaurants,
    "tag_code": tags
}
rel_restaurant_tag = pd.DataFrame(df_dict)

rel_restaurant_tag.head()

,restaurant_code,tag_code
0,RES0000,TAG0036
1,RES0000,TAG0054
2,RES0000,TAG0024
3,RES0000,TAG0065
4,RES0000,TAG0061


In [15]:
# rel_review_tag
reviews = []
tags = []

for rev, tdata in zip(review_df["review_code"], review_df["keywords"]):
    if pd.isna(tdata):
        continue

    tlist = tdata.split(",")
    for t in tlist:
        reviews.append(rev)
        tags.append(pipe(tag_df, "tag_name", "tag_code", t))

df_dict = {
    "review_code": reviews,
    "tag_code": tags
}
rel_review_tag = pd.DataFrame(df_dict)

rel_review_tag.head()

,review_code,tag_code
0,REV0000,TAG0065
1,REV0000,TAG0012
2,REV0000,TAG0105
3,REV0000,TAG0003
4,REV0000,TAG0036


## 각 테이블 최종 확인
- 사용한 데이터 컬럼 drop
- 컬럼명 erd 기준으로 통일
- 컬럼 순서 erd 기준으로 정돈
- 결측치 "" 처리
- to_csv()

In [16]:
# drop unnecessary
buff_df = restaurant_df.drop(columns=["category", "character", "tags"])

# rename columns
buff_df = buff_df.rename(columns={
    "restaurant_name":"name",
    "latitude":"lat",
    "longitude":"lng"
})

# order columns
cols = ["restaurant_code", "name", "img_link", "region", "address", "lat", "lng", "open_time", "close_time", "tel_no"]
buff_df = buff_df[cols]

# fill nan
buff_df = buff_df.fillna("")

# export to csv
restaurant_df = buff_df
restaurant_df.to_csv("db_csv_tablewise/restaurant.csv", index=False)
restaurant_df.head()

,restaurant_code,name,img_link,region,address,lat,lng,open_time,close_time,tel_no
0,RES0000,유태우스시,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,서울특별시 동작구 보라매로 113 1층 유태우스시,37.499463,126.927897,11:00,22:00,0507-1358-1246
1,RES0001,듀윗,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,서울특별시 동작구 대방동1길 10 1층,37.500364,126.926712,08:00,21:00,0507-2093-6672
2,RES0002,한제소곱창전골 신대방삼거리역점,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,"서울특별시 동작구 여의대방로22길 121 상가동 1층 1444호, 1445호",37.499106,126.926924,16:00,00:00,0507-1359-9161
3,RES0003,프로스퍼,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,서울특별시 동작구 여의대방로24길 101 1층,37.499088,126.925611,10:00,00:00,0507-1361-3276
4,RES0004,일진아구찜,https://d12zq4w4guyljn.cloudfront.net/300_300_...,신대방삼거리역,서울특별시 동작구 여의대방로24길 93 1층,37.498954,126.925189,11:00,22:00,02-817-5554


In [17]:
buff_df = review_df

# drop unnecessary
buff_df = buff_df.drop(columns="keywords")

# rename_column
buff_df = buff_df.rename(columns={
    "reviewer_code": "user_code",
    "reviewer_score": "score",
    "review_content": "content",
    "review_menu": "menu"
})

# order columns

# fill nan
buff_df = buff_df.fillna("")

# export to csv
review_df = buff_df
review_df.to_csv("db_csv_tablewise/review.csv", index=False)
review_df.head()

,review_code,restaurant_code,user_code,score,taste_level,price_level,service_level,content,menu
0,REV0000,RES0000,USR0000,4점,2.0,2.0,1.0,요즘 회전초밥집 정말 비싼데 여긴 가성비 갑이었어요! 위치는 조금 복잡한 곳에 있긴...,회전초밥
1,REV0001,RES0000,USR0001,4점,1.0,2.0,2.0,한접시에 1500원이면 저렴해요 근데 맛도 생각보다 괜찮았어요! 엄청난 퀄리티는 아...,스시
2,REV0002,RES0000,USR0002,5점,2.0,2.0,2.0,생일날 갔어요. 저녁시간 웨이팅있었지만 금방 자리에 앉았어요. 회전초밥 모든접시가 ...,회전초밥 1500원
3,REV0003,RES0000,USR0003,5점,2.0,2.0,2.0,"모든접시 1,700원 이라는 가성비에 맛도 좋고 이런 굿굿!!","연어,장어"
4,REV0004,RES0000,USR0004,5점,2.0,2.0,2.0,1500원이라가성비좋아요 다양하게 먹을 수 있고 여러가지종류있어서 좋아요,회전초밥


In [18]:
buff_df = menu_df

# drop unnecessary

# rename_column
buff_df = buff_df.rename(columns={
    "menu_name": "name"
})

# order columns

# fill nan
buff_df = buff_df.fillna("")

# export to csv
menu_df = buff_df
menu_df.to_csv("db_csv_tablewise/menu.csv", index=False)
menu_df.head()

,menu_code,restaurant_code,name,price,description
0,MEN0000,RES0000,회전초밥,,
1,MEN0001,RES0000,장어초밥,,
2,MEN0002,RES0001,프레지에 1호 홀케이크,50000.0,
3,MEN0003,RES0001,프레지에,10500.0,
4,MEN0004,RES0001,페르디셔 압펠,6000.0,


In [19]:
category_df.to_csv("db_csv_tablewise/category.csv", index=False)
category_df.head()

,category_code,category_name
0,CAT0000,고깃집
1,CAT0001,짜장면
2,CAT0002,일식당
3,CAT0003,숙성회
4,CAT0004,돼지갈비


In [20]:
buff_df = tag_df

# rename_column
buff_df = buff_df.rename(columns={
    "tag_name": "name"
})

tag_df = buff_df
tag_df.to_csv("db_csv_tablewise/tag.csv", index=False)
tag_df.head()

,tag_code,name
0,TAG0000,24시간영업
1,TAG0001,회식
2,TAG0002,포장전문
3,TAG0003,이국적/이색적
4,TAG0004,지역주민이 찾는


In [21]:
buff_df = user_df

# rename_column
buff_df = buff_df.rename(columns={
    "reviewer_avg_score": "avg_score",
    "reviewer_rv_cnt": "rv_cnt",
    "reviewer_follower_cnt": "follower_cnt"
})

user_df = buff_df
user_df.to_csv("db_csv_tablewise/user.csv", index=False)
user_df.head()

,user_code,name,avg_score,rv_cnt,follower_cnt
0,USR0000,냐밍,4.3,355,69
1,USR0001,발통이,4.5,113,1
2,USR0002,우리의나날,4.5,420,2
3,USR0003,냠냠냠히,4.8,71,0
4,USR0004,밍민,4.8,232,2


In [22]:
rel_restaurant_category.to_csv("db_csv_tablewise/rel_res_cat.csv", index=False)
rel_restaurant_tag.to_csv("db_csv_tablewise/rel_res_tag.csv", index=False)
rel_review_tag.to_csv("db_csv_tablewise/rel_rev_tag.csv", index=False)